In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load data

In [2]:
print("Loading data...")
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Loading data...
Train shape: (577347, 12)
Test shape: (247435, 11)


# Separate features and target

In [3]:
X_train = train_df.drop(['class', 'id'], axis=1)
y_train = train_df['class']
X_test = test_df.drop(['id'], axis=1)
test_ids = test_df['id']

# Encode categorical features

In [4]:
print("\nEncoding categorical features...")
le_spectral = LabelEncoder()
le_galaxy = LabelEncoder()
le_class = LabelEncoder()

X_train['spectral_type'] = le_spectral.fit_transform(X_train['spectral_type'])
X_train['galaxy_population'] = le_galaxy.fit_transform(X_train['galaxy_population'])
y_train_encoded = le_class.fit_transform(y_train)

X_test['spectral_type'] = le_spectral.transform(X_test['spectral_type'])
X_test['galaxy_population'] = le_galaxy.transform(X_test['galaxy_population'])

print(f"Encoded features - Spectral type: {le_spectral.classes_}")
print(f"Encoded features - Galaxy population: {le_galaxy.classes_}")
print(f"Encoded classes: {le_class.classes_}")


Encoding categorical features...
Encoded features - Spectral type: ['A/F' 'G/K' 'M' 'O/B']
Encoded features - Galaxy population: ['Blue_Cloud' 'Red_Sequence']
Encoded classes: ['GALAXY' 'QSO' 'STAR']


# Feature engineering

In [5]:
print("\nEngineering features...")
# Color indices (used in astronomy for classification)
X_train['u_g'] = X_train['u'] - X_train['g']
X_train['g_r'] = X_train['g'] - X_train['r']
X_train['r_i'] = X_train['r'] - X_train['i']
X_train['i_z'] = X_train['i'] - X_train['z']
X_train['u_r'] = X_train['u'] - X_train['r']

X_test['u_g'] = X_test['u'] - X_test['g']
X_test['g_r'] = X_test['g'] - X_test['r']
X_test['r_i'] = X_test['r'] - X_test['i']
X_test['i_z'] = X_test['i'] - X_test['z']
X_test['u_r'] = X_test['u'] - X_test['r']

# Magnitude ratios
X_train['mag_ratio_ug'] = X_train['u'] / (X_train['g'] + 1e-5)
X_train['mag_ratio_gr'] = X_train['g'] / (X_train['r'] + 1e-5)
X_train['mag_ratio_ri'] = X_train['r'] / (X_train['i'] + 1e-5)

X_test['mag_ratio_ug'] = X_test['u'] / (X_test['g'] + 1e-5)
X_test['mag_ratio_gr'] = X_test['g'] / (X_test['r'] + 1e-5)
X_test['mag_ratio_ri'] = X_test['r'] / (X_test['i'] + 1e-5)

# Mean magnitude
X_train['mean_mag'] = X_train[['u', 'g', 'r', 'i', 'z']].mean(axis=1)
X_test['mean_mag'] = X_test[['u', 'g', 'r', 'i', 'z']].mean(axis=1)

# Magnitude variance
X_train['mag_var'] = X_train[['u', 'g', 'r', 'i', 'z']].var(axis=1)
X_test['mag_var'] = X_test[['u', 'g', 'r', 'i', 'z']].var(axis=1)

print(f"Features after engineering: {X_train.shape[1]}")
print(f"Feature names: {list(X_train.columns)}")


Engineering features...
Features after engineering: 20
Feature names: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'mag_ratio_ug', 'mag_ratio_gr', 'mag_ratio_ri', 'mean_mag', 'mag_var']


# Split training data for validation

In [6]:
print("\nSplitting training data...")
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train_encoded, test_size=0.2, random_state=42, stratify=y_train_encoded
)


Splitting training data...


# Create XGBoost datasets

In [7]:
print("\nCreating XGBoost datasets...")
dtrain = xgb.DMatrix(X_train_split, label=y_train_split)
dval = xgb.DMatrix(X_val_split, label=y_val_split)
dtest = xgb.DMatrix(X_test)


Creating XGBoost datasets...


# XGBoost parameters optimized for GPU

In [8]:
params = {
    'objective': 'multi:softmax',
    'num_class': len(le_class.classes_),
    'tree_method': 'hist',  # GPU-accelerated histogram method
    'device': 'cuda',  # Updated for XGBoost 3.1+
    'max_depth': 16,
    'learning_rate': 0.01,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'min_child_weight': 2,
    'gamma': 0,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'scale_pos_weight': 1,
    'random_state': 42,
    'seed': 42,
    'eval_metric': 'mlogloss'
}

# Train model with early stopping

In [9]:
print("\nTraining XGBoost model on GPU...")
evals = [(dtrain, 'train'), (dval, 'validation')]
evals_result = {}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=24000,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=24000,
    verbose_eval=20
)

print(f"\nModel trained with {model.best_iteration + 1} rounds")
print(f"Best validation score: {evals_result['validation']['mlogloss'][model.best_iteration]:.6f}")


Training XGBoost model on GPU...
[0]	train-mlogloss:0.86677	validation-mlogloss:0.86706
[20]	train-mlogloss:0.67057	validation-mlogloss:0.67662
[40]	train-mlogloss:0.53730	validation-mlogloss:0.54787
[60]	train-mlogloss:0.43986	validation-mlogloss:0.45423
[80]	train-mlogloss:0.36517	validation-mlogloss:0.38283
[100]	train-mlogloss:0.30747	validation-mlogloss:0.32812
[120]	train-mlogloss:0.26115	validation-mlogloss:0.28463
[140]	train-mlogloss:0.22393	validation-mlogloss:0.24999
[160]	train-mlogloss:0.19403	validation-mlogloss:0.22256
[180]	train-mlogloss:0.16927	validation-mlogloss:0.20011
[200]	train-mlogloss:0.14891	validation-mlogloss:0.18204
[220]	train-mlogloss:0.13213	validation-mlogloss:0.16739
[240]	train-mlogloss:0.11828	validation-mlogloss:0.15561
[260]	train-mlogloss:0.10638	validation-mlogloss:0.14573
[280]	train-mlogloss:0.09627	validation-mlogloss:0.13751
[300]	train-mlogloss:0.08779	validation-mlogloss:0.13079
[320]	train-mlogloss:0.08041	validation-mlogloss:0.12518
[34

# Make predictions on test set

In [10]:
print("\nMaking predictions on test set...")
y_pred_probs = model.predict(dtest)
y_pred = y_pred_probs.astype(int)


Making predictions on test set...


# Decode predictions back to class names

In [11]:
y_pred_classes = le_class.inverse_transform(y_pred)

# Create submission file

In [12]:
print("\nCreating submission file...")
submission_df = pd.DataFrame({
    'id': test_ids,
    'class': y_pred_classes
})

submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file created: submission.csv")
print(f"Submission shape: {submission_df.shape}")
print("\nFirst 10 rows of submission:")
print(submission_df.head(10))
print("\nLast 10 rows of submission:")
print(submission_df.tail(10))


Creating submission file...

Submission file created: submission.csv
Submission shape: (247435, 2)

First 10 rows of submission:
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
5  577352  GALAXY
6  577353  GALAXY
7  577354    STAR
8  577355  GALAXY
9  577356  GALAXY

Last 10 rows of submission:
            id   class
247425  824772    STAR
247426  824773  GALAXY
247427  824774  GALAXY
247428  824775     QSO
247429  824776  GALAXY
247430  824777     QSO
247431  824778     QSO
247432  824779  GALAXY
247433  824780     QSO
247434  824781  GALAXY


# Display class distribution in predictions

In [13]:
print("\nClass distribution in predictions:")
print(submission_df['class'].value_counts())


Class distribution in predictions:
class
GALAXY    162307
QSO        50013
STAR       35115
Name: count, dtype: int64


# Feature importance

In [14]:
print("\nTop 20 most important features:")
importance = model.get_score(importance_type='weight')
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
for i, (feat, score) in enumerate(sorted_importance[:20], 1):
    print(f"{i}. {feat}: {score}")

print("\nModel training and prediction complete!")


Top 20 most important features:
1. delta: 1810273.0
2. alpha: 1762475.0
3. i_z: 1580323.0
4. redshift: 1447552.0
5. r_i: 1087163.0
6. mag_ratio_ri: 1026762.0
7. mag_var: 975923.0
8. g_r: 953162.0
9. z: 937763.0
10. mag_ratio_gr: 926793.0
11. u_r: 914396.0
12. u_g: 903195.0
13. mag_ratio_ug: 884756.0
14. g: 858852.0
15. u: 856620.0
16. i: 799233.0
17. r: 795493.0
18. mean_mag: 725237.0
19. spectral_type: 85706.0
20. galaxy_population: 10166.0

Model training and prediction complete!
